In [0]:
# ============================================================
# OBJETIVO:
#   1. Leer las tablas Bronze.
#   2. Reemplazar valores "?" por NULL.
#   3. Convertir columnas a sus tipos correctos.
#   4. Normalizar campos de texto.
#   5. Eliminar duplicados.
#   6. Unir automóviles con fabricantes mediante "make".
#   7. Conservar la trazabilidad proveniente de _metadata.
#   8. Crear la tabla Delta Silver.
# ============================================================


# ------------------------------------------------------------
# 1. IMPORTAR FUNCIONES
# ------------------------------------------------------------

from pyspark.sql.functions import (
    col,
    when,
    trim,
    lower,
    current_timestamp,
    lit,
    row_number
)

from pyspark.sql.window import Window


# ------------------------------------------------------------
# 2. PARÁMETROS
# ------------------------------------------------------------

catalog_name = "catalog"

bronze_schema = "bronze"
silver_schema = "silver"

automobile_bronze_table = (
    f"{catalog_name}.{bronze_schema}.automobile_raw"
)

manufacturers_bronze_table = (
    f"{catalog_name}.{bronze_schema}."
    f"automobile_manufacturers_raw"
)

silver_table = (
    f"{catalog_name}.{silver_schema}.automobile_enriched"
)

print("Tabla Bronze de automóviles:")
print(automobile_bronze_table)

print("\nTabla Bronze de fabricantes:")
print(manufacturers_bronze_table)

print("\nTabla Silver de destino:")
print(silver_table)

Tabla Bronze de automóviles:
catalog.bronze.automobile_raw

Tabla Bronze de fabricantes:
catalog.bronze.automobile_manufacturers_raw

Tabla Silver de destino:
catalog.silver.automobile_enriched


In [0]:
# Leer las tablas Bronze


df_automobile_bronze = spark.table(
    automobile_bronze_table
)

df_manufacturers_bronze = spark.table(
    manufacturers_bronze_table
)

automobile_bronze_count = (
    df_automobile_bronze.count()
)

manufacturers_bronze_count = (
    df_manufacturers_bronze.count()
)

print(
    "Registros Automobile Bronze:",
    automobile_bronze_count
)

print(
    "Registros Manufacturers Bronze:",
    manufacturers_bronze_count
)

assert automobile_bronze_count > 0, (
    "La tabla Bronze automobile_raw está vacía."
)

assert manufacturers_bronze_count > 0, (
    "La tabla Bronze de fabricantes está vacía."
)

Registros Automobile Bronze: 205
Registros Manufacturers Bronze: 22


In [0]:
# ------------------------------------------------------------
# 5. IDENTIFICAR COLUMNAS TÉCNICAS
# ------------------------------------------------------------

technical_columns = [
    "source_file",
    "source_file_name",
    "source_file_size",
    "source_file_modification_time",
    "source_system",
    "source_dataset",
    "ingestion_timestamp",
    "record_hash"
]

automobile_business_columns = [
    column_name
    for column_name in df_automobile_bronze.columns
    if column_name not in technical_columns
]

manufacturer_business_columns = [
    column_name
    for column_name in df_manufacturers_bronze.columns
    if column_name not in technical_columns
]

print("Columnas de negocio de automóviles:")
print(automobile_business_columns)

print("\nColumnas de negocio de fabricantes:")
print(manufacturer_business_columns)

Columnas de negocio de automóviles:
['symboling', 'normalized_losses', 'make', 'fuel_type', 'aspiration', 'num_of_doors', 'body_style', 'drive_wheels', 'engine_location', 'wheel_base', 'length', 'width', 'height', 'curb_weight', 'engine_type', 'num_of_cylinders', 'engine_size', 'fuel_system', 'bore', 'stroke', 'compression_ratio', 'horsepower', 'peak_rpm', 'city_mpg', 'highway_mpg', 'price']

Columnas de negocio de fabricantes:
['manufacturer_id', 'make', 'vehicle_count', 'avg_price', 'avg_horsepower', 'avg_city_mpg', 'avg_highway_mpg', 'dominant_fuel_type', 'dominant_body_style']


In [0]:
# ------------------------------------------------------------
# 6. REEMPLAZAR ?, CADENAS VACÍAS Y ESPACIOS POR NULL
# ------------------------------------------------------------

df_automobile_clean = df_automobile_bronze

for column_name in automobile_business_columns:

    df_automobile_clean = (
        df_automobile_clean
        .withColumn(
            column_name,
            when(
                trim(col(column_name).cast("string")).isin(
                    "?",
                    "",
                    "null",
                    "NULL"
                ),
                None
            ).otherwise(
                trim(col(column_name).cast("string"))
            )
        )
    )


df_manufacturers_clean = df_manufacturers_bronze

for column_name in manufacturer_business_columns:

    df_manufacturers_clean = (
        df_manufacturers_clean
        .withColumn(
            column_name,
            when(
                trim(col(column_name).cast("string")).isin(
                    "?",
                    "",
                    "null",
                    "NULL"
                ),
                None
            ).otherwise(
                trim(col(column_name).cast("string"))
            )
        )
    )

print("Valores ?, vacíos y NULL de texto normalizados.")

Valores ?, vacíos y NULL de texto normalizados.


In [0]:
# ------------------------------------------------------------
# 7. CONVERTIR TIPOS DEL DATASET AUTOMOBILE
# ------------------------------------------------------------

automobile_integer_columns = [
    "symboling",
    "normalized_losses",
    "curb_weight",
    "engine_size",
    "horsepower",
    "peak_rpm",
    "city_mpg",
    "highway_mpg",
    "price"
]

automobile_decimal_columns = [
    "wheel_base",
    "length",
    "width",
    "height",
    "bore",
    "stroke",
    "compression_ratio"
]

for column_name in automobile_integer_columns:

    df_automobile_clean = (
        df_automobile_clean
        .withColumn(
            column_name,
            col(column_name).cast("integer")
        )
    )


for column_name in automobile_decimal_columns:

    df_automobile_clean = (
        df_automobile_clean
        .withColumn(
            column_name,
            col(column_name).cast("double")
        )
    )

print("Tipos numéricos de Automobile_data convertidos.")

Tipos numéricos de Automobile_data convertidos.


In [0]:
# ------------------------------------------------------------
# 8. CONVERTIR TIPOS DEL DATASET DE FABRICANTES
# ------------------------------------------------------------

manufacturer_integer_columns = [
    "manufacturer_id",
    "vehicle_count"
]

manufacturer_decimal_columns = [
    "avg_price",
    "avg_horsepower",
    "avg_city_mpg",
    "avg_highway_mpg"
]

for column_name in manufacturer_integer_columns:

    df_manufacturers_clean = (
        df_manufacturers_clean
        .withColumn(
            column_name,
            col(column_name).cast("integer")
        )
    )


for column_name in manufacturer_decimal_columns:

    df_manufacturers_clean = (
        df_manufacturers_clean
        .withColumn(
            column_name,
            col(column_name).cast("double")
        )
    )

print("Tipos numéricos de fabricantes convertidos.")

Tipos numéricos de fabricantes convertidos.


In [0]:
# ------------------------------------------------------------
# 10. ELIMINAR DUPLICADOS
# ------------------------------------------------------------

automobile_duplicate_columns = [
    "symboling",
    "normalized_losses",
    "make",
    "fuel_type",
    "aspiration",
    "num_of_doors",
    "body_style",
    "drive_wheels",
    "engine_location",
    "wheel_base",
    "length",
    "width",
    "height",
    "curb_weight",
    "engine_type",
    "num_of_cylinders",
    "engine_size",
    "fuel_system",
    "bore",
    "stroke",
    "compression_ratio",
    "horsepower",
    "peak_rpm",
    "city_mpg",
    "highway_mpg",
    "price"
]

automobile_before_duplicates = (
    df_automobile_clean.count()
)

df_automobile_deduplicated = (
    df_automobile_clean
    .dropDuplicates(
        automobile_duplicate_columns
    )
)

automobile_after_duplicates = (
    df_automobile_deduplicated.count()
)

duplicates_removed = (
    automobile_before_duplicates
    - automobile_after_duplicates
)

df_manufacturers_deduplicated = (
    df_manufacturers_clean
    .dropDuplicates(
        [
            "manufacturer_id",
            "make"
        ]
    )
)

print(
    "Registros antes de eliminar duplicados:",
    automobile_before_duplicates
)

print(
    "Registros después de eliminar duplicados:",
    automobile_after_duplicates
)

print(
    "Duplicados eliminados:",
    duplicates_removed
)

Registros antes de eliminar duplicados: 205
Registros después de eliminar duplicados: 205
Duplicados eliminados: 0


In [0]:
# ------------------------------------------------------------
# 11. PREPARAR EL CATÁLOGO DE FABRICANTES
# ------------------------------------------------------------

df_manufacturer_catalog = (
    df_manufacturers_deduplicated
    .select(
        "manufacturer_id",
        "make",

        col("vehicle_count").alias(
            "source_vehicle_count"
        ),

        col("avg_price").alias(
            "source_average_price"
        ),

        col("avg_horsepower").alias(
            "source_average_horsepower"
        ),

        col("avg_city_mpg").alias(
            "source_average_city_mpg"
        ),

        col("avg_highway_mpg").alias(
            "source_average_highway_mpg"
        ),

        col("dominant_fuel_type").alias(
            "manufacturer_dominant_fuel_type"
        ),

        col("dominant_body_style").alias(
            "manufacturer_dominant_body_style"
        )
    )
)

display(
    df_manufacturer_catalog
    .orderBy("manufacturer_id")
)

manufacturer_id,make,source_vehicle_count,source_average_price,source_average_horsepower,source_average_city_mpg,source_average_highway_mpg,manufacturer_dominant_fuel_type,manufacturer_dominant_body_style
1,alfa-romero,3,15498.33,125.33,20.33,26.67,gas,convertible
2,audi,7,17859.17,121.0,18.86,24.14,gas,sedan
3,bmw,8,26118.75,138.88,19.38,25.38,gas,sedan
4,chevrolet,3,6007.0,62.67,41.0,46.33,gas,hatchback
5,dodge,9,7875.44,86.33,28.0,34.11,gas,hatchback
6,honda,13,8184.69,80.23,30.38,35.46,gas,hatchback
7,isuzu,4,8916.5,77.0,31.0,36.0,gas,sedan
8,jaguar,3,34600.0,204.67,14.33,18.33,gas,sedan
9,mazda,17,10652.88,85.53,25.71,31.94,gas,hatchback
10,mercedes-benz,8,33647.0,146.25,18.5,21.0,diesel,sedan


In [0]:
# ------------------------------------------------------------
# 13. UNIR AUTOMÓVILES CON FABRICANTES
# ------------------------------------------------------------

df_silver_joined = (
    df_automobile_deduplicated.alias("auto")
    .join(
        df_manufacturer_catalog.alias("manufacturer"),
        on="make",
        how="left"
    )
)


# ------------------------------------------------------------
# 14. AGREGAR BANDERAS DE CALIDAD
# ------------------------------------------------------------

df_silver_joined = (
    df_silver_joined

    .withColumn(
        "is_manufacturer_matched",
        col("manufacturer_id").isNotNull()
    )

    .withColumn(
        "is_price_available",
        col("price").isNotNull()
    )

    .withColumn(
        "is_horsepower_available",
        col("horsepower").isNotNull()
    )

    .withColumn(
        "is_mpg_available",
        col("city_mpg").isNotNull()
        & col("highway_mpg").isNotNull()
    )

    .withColumn(
        "data_quality_status",
        when(
            col("manufacturer_id").isNull(),
            lit("MISSING_MANUFACTURER")
        )
        .when(
            col("price").isNull(),
            lit("MISSING_PRICE")
        )
        .when(
            col("horsepower").isNull(),
            lit("MISSING_HORSEPOWER")
        )
        .otherwise(
            lit("VALID")
        )
    )

    .withColumn(
        "silver_transformation_timestamp",
        current_timestamp()
    )
)

In [0]:
# ------------------------------------------------------------
# 14. CREAR AUTOMOBILE_ID
# ------------------------------------------------------------

automobile_window = Window.orderBy(
    col("record_hash"),
    col("make"),
    col("price")
)

df_silver_with_id = (
    df_silver_joined
    .withColumn(
        "automobile_id",
        row_number().over(
            automobile_window
        )
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# ------------------------------------------------------------
# 15. SELECCIONAR COLUMNAS FINALES
# ------------------------------------------------------------

df_silver_final = (
    df_silver_with_id
    .select(
        # Identificadores
        "automobile_id",
        "manufacturer_id",

        # Características generales
        "make",
        "symboling",
        "normalized_losses",
        "fuel_type",
        "aspiration",
        "num_of_doors",
        "body_style",
        "drive_wheels",
        "engine_location",

        # Dimensiones
        "wheel_base",
        "length",
        "width",
        "height",
        "curb_weight",

        # Motor
        "engine_type",
        "num_of_cylinders",
        "engine_size",
        "fuel_system",
        "bore",
        "stroke",
        "compression_ratio",
        "horsepower",
        "peak_rpm",

        # Rendimiento y precio
        "city_mpg",
        "highway_mpg",
        "price",

        # Información complementaria del fabricante
        "source_vehicle_count",
        "source_average_price",
        "source_average_horsepower",
        "source_average_city_mpg",
        "source_average_highway_mpg",
        "manufacturer_dominant_fuel_type",
        "manufacturer_dominant_body_style",

        # Calidad
        "is_manufacturer_matched",
        "is_price_available",
        "is_horsepower_available",
        "is_mpg_available",
        "data_quality_status",

        # Trazabilidad obtenida desde _metadata
        "source_system",
        "source_dataset",
        "source_file",
        "source_file_name",
        "source_file_size",
        "source_file_modification_time",
        "ingestion_timestamp",
        "silver_transformation_timestamp",
        "record_hash"
    )
)

display(
    df_silver_final.limit(10)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


automobile_id,manufacturer_id,make,symboling,normalized_losses,fuel_type,aspiration,num_of_doors,body_style,drive_wheels,engine_location,wheel_base,length,width,height,curb_weight,engine_type,num_of_cylinders,engine_size,fuel_system,bore,stroke,compression_ratio,horsepower,peak_rpm,city_mpg,highway_mpg,price,source_vehicle_count,source_average_price,source_average_horsepower,source_average_city_mpg,source_average_highway_mpg,manufacturer_dominant_fuel_type,manufacturer_dominant_body_style,is_manufacturer_matched,is_price_available,is_horsepower_available,is_mpg_available,data_quality_status,source_system,source_dataset,source_file,source_file_name,source_file_size,source_file_modification_time,ingestion_timestamp,silver_transformation_timestamp,record_hash
1,2,audi,2,null,gas,std,two,sedan,fwd,front,99.8,177.3,66.3,53.1,2507,ohc,five,136,mpfi,3.19,3.4,8.5,110,5500,19,25,15250,7,17859.17,121.0,18.86,24.14,gas,sedan,true,true,true,true,VALID,ADLS_GEN2,Automobile_data.csv,abfss://raw@steacpauto01.dfs.core.windows.net/automobile/input/Automobile_data.csv,Automobile_data.csv,25070,2026-10-03T15:52:44.000Z,2026-10-03T17:28:25.293Z,2026-10-03T17:45:34.353Z,01a1f5886133dddba7e5a42d685cd4089f411afebedc0ad2f8d9e54b06f72593
2,20,toyota,-1,65,gas,std,four,sedan,fwd,front,102.4,175.6,66.5,54.9,2414,ohc,four,122,mpfi,3.31,3.54,8.7,92,4200,27,32,10898,32,9885.81,92.78,27.5,32.91,gas,hatchback,true,true,true,true,VALID,ADLS_GEN2,Automobile_data.csv,abfss://raw@steacpauto01.dfs.core.windows.net/automobile/input/Automobile_data.csv,Automobile_data.csv,25070,2026-10-03T15:52:44.000Z,2026-10-03T17:28:25.293Z,2026-10-03T17:45:34.353Z,0215654855fb1cf185862447a2c9c0d1140c58f9d07c39a5cdbc59eb592ef888
3,9,mazda,0,118,gas,std,four,sedan,rwd,front,104.9,175.0,66.1,54.4,2670,ohc,four,140,mpfi,3.76,3.16,8.0,120,5000,19,27,18280,17,10652.88,85.53,25.71,31.94,gas,hatchback,true,true,true,true,VALID,ADLS_GEN2,Automobile_data.csv,abfss://raw@steacpauto01.dfs.core.windows.net/automobile/input/Automobile_data.csv,Automobile_data.csv,25070,2026-10-03T15:52:44.000Z,2026-10-03T17:28:25.293Z,2026-10-03T17:45:34.353Z,028b10778518a408dcf6ced40e883f97077a9584a5c8f61e2d7eb04afa821aae
4,20,toyota,3,197,gas,std,two,hatchback,rwd,front,102.9,183.5,67.7,52.0,3016,dohc,six,171,mpfi,3.27,3.35,9.3,161,5200,19,24,15998,32,9885.81,92.78,27.5,32.91,gas,hatchback,true,true,true,true,VALID,ADLS_GEN2,Automobile_data.csv,abfss://raw@steacpauto01.dfs.core.windows.net/automobile/input/Automobile_data.csv,Automobile_data.csv,25070,2026-10-03T15:52:44.000Z,2026-10-03T17:28:25.293Z,2026-10-03T17:45:34.353Z,052fa4c05ee4ec2e8ccd4d6501c3f07f5511520818d5d5d8f0a4e0565bf04f62
5,19,subaru,0,89,gas,std,four,wagon,fwd,front,97.0,173.5,65.4,53.0,2290,ohcf,four,108,2bbl,3.62,2.64,9.0,82,4800,28,32,7463,12,8541.25,86.25,26.33,30.75,gas,sedan,true,true,true,true,VALID,ADLS_GEN2,Automobile_data.csv,abfss://raw@steacpauto01.dfs.core.windows.net/automobile/input/Automobile_data.csv,Automobile_data.csv,25070,2026-10-03T15:52:44.000Z,2026-10-03T17:28:25.293Z,2026-10-03T17:45:34.353Z,068f1470521a69d12faf55d669ca985508c35e286134addbff56246ac99d2403
6,6,honda,0,110,gas,std,four,sedan,fwd,front,96.5,163.4,64.0,54.5,2010,ohc,four,92,1bbl,2.91,3.41,9.2,76,6000,30,34,7295,13,8184.69,80.23,30.38,35.46,gas,hatchback,true,true,true,true,VALID,ADLS_GEN2,Automobile_data.csv,abfss://raw@steacpauto01.dfs.core.windows.net/automobile/input/Automobile_data.csv,Automobile_data.csv,25070,2026-10-03T15:52:44.000Z,2026-10-03T17:28:25.293Z,2026-10-03T17:45:34.353Z,076b0f786b2d3935e8026b8dcb219de49aaa6a090ff39089fa0bd32fece4007e
7,15,plymouth,1,154,gas,std,four,sedan,fwd,front,93.7,167.3,63.8,50.8,2191,ohc,four,98,2bbl,2.97,3.23,9.4,68,5500,31,38,7609,7,7963.43,86.71,28.14,34.14,gas,hatchback,true,true,true,true,VALID,ADLS_GEN2,Automobile_data.csv,abfss://raw@steacpauto01.dfs.core.windows.net/automobile/input/Automobile_data.csv,Automobile_data.csv,25070,2026-10-03T15:52:44.000Z,2026-10-03T17:28:25.293Z,2026-10-03T17:45:34.353Z

In [0]:
# ------------------------------------------------------------
# 16. CREAR TABLA DELTA SILVER
# ------------------------------------------------------------

(
    df_silver_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

print(
    f"Tabla Silver creada correctamente: "
    f"{silver_table}"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Tabla Silver creada correctamente: catalog.silver.automobile_enriched


In [0]:
# ------------------------------------------------------------
# 17. VALIDACIONES
# ------------------------------------------------------------

df_silver_validation = spark.table(
    silver_table
)

silver_count = (
    df_silver_validation.count()
)

missing_manufacturer_count = (
    df_silver_validation
    .filter(
        col("manufacturer_id").isNull()
    )
    .count()
)

missing_price_count = (
    df_silver_validation
    .filter(
        col("price").isNull()
    )
    .count()
)

missing_horsepower_count = (
    df_silver_validation
    .filter(
        col("horsepower").isNull()
    )
    .count()
)

valid_record_count = (
    df_silver_validation
    .filter(
        col("data_quality_status") == "VALID"
    )
    .count()
)

assert silver_count > 0, (
    "La tabla Silver está vacía."
)

assert missing_manufacturer_count == 0, (
    f"Existen {missing_manufacturer_count} registros "
    f"sin fabricante relacionado."
)

print()
print("Transformación Silver terminada correctamente.")
print("=============================================")

print(
    f"Registros en Bronze: "
    f"{automobile_bronze_count}"
)

print(
    f"Duplicados eliminados: "
    f"{duplicates_removed}"
)

print(
    f"Registros en Silver: "
    f"{silver_count}"
)

print(
    f"Registros válidos: "
    f"{valid_record_count}"
)

print(
    f"Registros sin fabricante: "
    f"{missing_manufacturer_count}"
)

print(
    f"Registros sin precio: "
    f"{missing_price_count}"
)

print(
    f"Registros sin horsepower: "
    f"{missing_horsepower_count}"
)


Transformación Silver terminada correctamente.
Registros en Bronze: 205
Duplicados eliminados: 0
Registros en Silver: 205
Registros válidos: 199
Registros sin fabricante: 0
Registros sin precio: 4
Registros sin horsepower: 2


In [0]:
# ------------------------------------------------------------
# 18. MOSTRAR RESULTADOS SILVER
# ------------------------------------------------------------

display(
    spark.sql(f"""
        SELECT
            automobile_id,
            manufacturer_id,
            make,
            fuel_type,
            body_style,
            engine_size,
            horsepower,
            city_mpg,
            highway_mpg,
            price,
            data_quality_status,
            source_file_name,
            ingestion_timestamp,
            silver_transformation_timestamp
        FROM {silver_table}
        ORDER BY automobile_id
        LIMIT 25
    """)
)

automobile_id,manufacturer_id,make,fuel_type,body_style,engine_size,horsepower,city_mpg,highway_mpg,price,data_quality_status,source_file_name,ingestion_timestamp,silver_transformation_timestamp
1,2,audi,gas,sedan,136,110,19,25,15250,VALID,Automobile_data.csv,2026-10-03T17:28:25.293Z,2026-10-03T17:45:50.791Z
2,20,toyota,gas,sedan,122,92,27,32,10898,VALID,Automobile_data.csv,2026-10-03T17:28:25.293Z,2026-10-03T17:45:50.791Z
3,9,mazda,gas,sedan,140,120,19,27,18280,VALID,Automobile_data.csv,2026-10-03T17:28:25.293Z,2026-10-03T17:45:50.791Z
4,20,toyota,gas,hatchback,171,161,19,24,15998,VALID,Automobile_data.csv,2026-10-03T17:28:25.293Z,2026-10-03T17:45:50.791Z
5,19,subaru,gas,wagon,108,82,28,32,7463,VALID,Automobile_data.csv,2026-10-03T17:28:25.293Z,2026-10-03T17:45:50.791Z
6,6,honda,gas,sedan,92,76,30,34,7295,VALID,Automobile_data.csv,2026-10-03T17:28:25.293Z,2026-10-03T17:45:50.791Z
7,15,plymouth,gas,sedan,98,68,31,38,7609,VALID,Automobile_data.csv,2026-10-03T17:28:25.293Z,2026-10-03T17:45:50.791Z
8,19,subaru,gas,hatchback,108,73,26,31,7603,VALID,Automobile_data.csv,2026-10-03T17:28:25.293Z,2026-10-03T17:45:50.791Z
9,20,toyota,gas,sedan,98,70,38,47,7738,VALID,Automobile_data.csv,2026-10-03T17:28:25.293Z,2026-10-03T17:45:50.791Z
10,6,honda,gas,wagon,92,76,30,34,7295,VALID,Automobile_data.csv,2026-10-03T17:28:25.293Z,2026-10-03T17:45:50.791Z


In [0]:
# ------------------------------------------------------------
# 19. RESUMEN DE CALIDAD
# ------------------------------------------------------------

display(
    spark.sql(f"""
        SELECT
            data_quality_status,
            COUNT(*) AS total_records
        FROM {silver_table}
        GROUP BY data_quality_status
        ORDER BY total_records DESC
    """)
)

data_quality_status,total_records
VALID,199
MISSING_PRICE,4
MISSING_HORSEPOWER,2
